# 03 · Use It From an Agent

**Module 11 · MCP Tool Servers**

Now the payoff. The same `shop_server.py` is used by **two different agents**, and neither contains a single line of tool code:

1. **Our own agent loop** (the one from Module 02), with the tools coming from a small MCP client.
2. **The OpenAI Agents SDK**, which has MCP support built in.

> About 10 small OpenAI calls.

In [1]:
import subprocess
import sys

import mcp_desk
from toolbox import Toolbox, run_sync

print("Ready")

Ready


## 1. Our own agent, with tools from the server
The client side is a small class, `Toolbox`. Read how it does the three jobs (connect, discover, forward):

*From `toolbox.py`:*

```python
class Toolbox:
    def __init__(self, gate=None):
        self.stack = AsyncExitStack()   # closes every connection when we leave the 'async with'
        self.gate = gate                # an optional security check (see mcp_security.py)
        self.clients = {}               # server name -> its connected client
        self.owner = {}                 # tool name -> the name of the server that offers it
        self.tools = []                 # the tools we show to the model
        self.rejected = []              # tools we refused to show, with the reason
        self.calls = []                 # every call that was made: (server, tool, arguments)

    async def __aenter__(self):
        return self

    async def __aexit__(self, error_type, error, traceback):
        await self.stack.aclose()

    async def add_server(self, name, script, extra_env=None, trusted=False):
        """Start a server, ask what tools it offers, and keep the ones that pass the gate.
        A TRUSTED server is one we wrote and control (like our own shop server): its tools skip the gate."""
        client = await self.stack.enter_async_context(open_client(script, extra_env))
        self.clients[name] = client

        listed = await client.list_tools()
        offered = listed.tools
        if self.gate is not None and not trusted:
            offered, problems = self.gate.filter(name, listed.tools)
            for problem in problems:
                self.rejected.append(problem)

        for tool in offered:
            if tool.name in self.owner:
                self.rejected.append(name + "/" + tool.name + ": REJECTED, another server already offers a tool with this name")
                continue
            self.owner[tool.name] = name
            self.tools.append(tool)
            if trusted and self.gate is not None:
                # remember the names of OUR tools: a third-party description that mentions them is trying to steer them
                self.gate.known_tool_names = list(self.gate.known_tool_names) + [tool.name]
        return offered

    def openai_tools(self):
        """The tools in the format the OpenAI chat API wants. The MCP input schema is already JSON Schema, so we reuse it."""
        converted = []
        for tool in self.tools:
            converted.append({
                "type": "function",
                "function": {"name": tool.name, "description": tool.description, "parameters": tool.input_schema},
            })
        return converted

    async def call(self, tool_name, arguments):
        """Forward a tool call to the server that owns the tool. Returns the result as text."""
        if tool_name not in self.owner:
            return "Unknown tool: " + tool_name

        server = self.owner[tool_name]
        if self.gate is not None:
            allowed, reason = self.gate.allow_call(server, tool_name, arguments)
            if not allowed:
                self.calls.append((server, tool_name, arguments, "BLOCKED"))
                return "Blocked: " + reason

        result = await self.clients[server].call_tool(tool_name, arguments)
        text = text_of(result)
        outcome = "ok"
        if result.is_error:
            outcome = "error"
        self.calls.append((server, tool_name, arguments, outcome))
        return text
```

And the agent loop. Compare it with the one in Module 02. The loop is the same. The only difference is that the tools are not defined here: they come from `box.openai_tools()`, and a tool call is forwarded with `await box.call(...)`:

*From `mcp_desk.py`:*

```python
async def answer(email, box, max_steps=6, verbose=False):
    """The agent loop from Module 02, with ONE change: the tools come from the toolbox, not from our own code."""
    messages = [
        {"role": "system", "content": INSTRUCTIONS},
        {"role": "user", "content": email},
    ]
    tools = box.openai_tools()      # discovered from the servers a moment ago

    for step in range(max_steps):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)
        reply = response.choices[0].message
        messages.append(reply)

        if not reply.tool_calls:
            return reply.content

        for tool_call in reply.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            if verbose:
                print("    tool: " + name + "(" + str(args) + ")")
            result = await box.call(name, args)      # forwarded to whichever server offers this tool
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    return "Stopped: too many steps."
```

In [2]:
async def three_emails():
    async with Toolbox() as box:
        await box.add_server("shop", "shop_server.py", trusted=True)
        names = []
        for tool in box.tools:
            names.append(tool.name)
        print("The desk discovered:", names, "\n")

        for email in ["Hi, where is my order A1002?",
                      "The left earbud on my headphones (order A1001) stopped working. Please refund it.",
                      "Please refund my 4K television, order A1004. I changed my mind."]:
            print("Customer:", email)
            reply = await mcp_desk.answer(email, box, verbose=True)
            print("Desk    :", reply.replace("\n", " "), "\n")

        print("What the server recorded:", await box.call("list_refunds", {}))


run_sync(three_emails)

The desk discovered: ['lookup_order', 'check_refund_eligibility', 'issue_refund', 'search_policies', 'list_refunds'] 

Customer: Hi, where is my order A1002?


    tool: lookup_order({'order_id': 'A1002'})


Desk    : Your order A1002, which is a pair of running shoes, is currently in transit and is expected to be delivered on November 26, 2026. 

Customer: The left earbud on my headphones (order A1001) stopped working. Please refund it.


    tool: lookup_order({'order_id': 'A1001'})


    tool: check_refund_eligibility({'order_id': 'A1001'})


    tool: issue_refund({'order_id': 'A1001'})


Desk    : A refund of $79.00 has been issued for your order A1001. You should see the amount reflected in your account soon. If you need further assistance, feel free to ask! 

Customer: Please refund my 4K television, order A1004. I changed my mind.


    tool: check_refund_eligibility({'order_id': 'A1004'})


    tool: issue_refund({'order_id': 'A1004'})


Desk    : Your order A1004 for the 4K television is eligible for a refund, but the amount of $250.00 requires approval from a person. The refund has not been issued yet. 

What the server recorded: {"issued": [{"order_id": "A1001", "amount": 79.0}], "waiting_for_a_person": ["A1004"]}


The desk has **no tool code and no tool descriptions**. The refund limit held, because it is in the server.

## 2. The same server in the OpenAI Agents SDK
Now a completely different piece of software. The SDK starts the server, discovers the tools and forwards the calls, all by itself. This is the whole program, `agents_sdk_mcp.py` (we run it as a separate program, as you would):

In [3]:
print(open("agents_sdk_mcp.py", encoding="utf-8").read().split("async def main():")[1].split("if __name__")[0])


    if hasattr(sys.stdout, "reconfigure"):      # (a notebook's output has no reconfigure)
        sys.stdout.reconfigure(encoding="utf-8")

    # One line starts the server and connects to it. The SDK discovers its tools by itself.
    async with MCPServerStdio(params={"command": sys.executable, "args": [SERVER]}, cache_tools_list=True) as server:
        agent = Agent(
            name="Support agent",
            instructions=("You are a customer support agent at an online shop. Use your tools. "
                          "Never say a refund is issued unless a tool says so. Answer in at most 3 short sentences."),
            mcp_servers=[server],
            model="gpt-4o-mini",
        )

        tools = await server.list_tools()
        names = []
        for tool in tools:
            names.append(tool.name)
        print("Tools the Agents SDK discovered: " + ", ".join(names) + "\n")

        for email in EMAILS:
            result = await Runner.run(agent, email)
            pr

In [4]:
result = subprocess.run([sys.executable, "agents_sdk_mcp.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-1500:])

Tools the Agents SDK discovered: lookup_order, check_refund_eligibility, issue_refund, search_policies, list_refunds

Customer: Hi, where is my order A1002?
Desk    : Your order A1002 for running shoes is currently in transit and is expected to be delivered on November 26, 2026.

Customer: The left earbud on my headphones (order A1001) stopped working. Please refund it.
Desk    : A refund of $79.00 has been issued for your wireless headphones (order A1001). If you need further assistance, feel free to ask!

Customer: Please refund my 4K television, order A1004. I changed my mind.
Desk    : Your refund request for the 4K television (order A1004) is eligible, but it is awaiting approval since it's above the automatic limit. It has not yet been issued. You will be notified once the refund is processed.




Same server file, same tools, same $100 limit. **We wrote the tools once.** That is the promise of MCP. (In Module 06 we wrapped the same functions three times, once per framework.)

## 3. The server grows, and the agent does not change
Add a tool to the server, and every client simply **finds** it. `shop_server.py` has one extra tool, `get_store_hours`, that appears when an environment setting is on. The agent code below is the same in both runs:

In [5]:
async def ask_about_hours(extra):
    environment = None
    if extra:
        environment = {"SHOP_EXTRA_TOOLS": "1"}

    async with Toolbox() as box:
        await box.add_server("shop", "shop_server.py", environment, trusted=True)
        names = []
        for tool in box.tools:
            names.append(tool.name)
        reply = await mcp_desk.answer("What are your phone support hours?", box)
        return names, reply


for extra in [False, True]:
    names, reply = run_sync(ask_about_hours, extra)
    print("Server has the extra tool:", extra)
    print("   tools :", names)
    print("   reply :", reply.replace("\n", " "))
    print()

Server has the extra tool: False
   tools : ['lookup_order', 'check_refund_eligibility', 'issue_refund', 'search_policies', 'list_refunds']
   reply : I couldn't find information about phone support hours. Please check our website or contact us via email for assistance.



Server has the extra tool: True
   tools : ['lookup_order', 'check_refund_eligibility', 'issue_refund', 'search_policies', 'list_refunds', 'get_store_hours']
   reply : Phone support is open Monday to Friday, from 9:00 to 17:00.



Without the tool, the agent does not know the hours (and should say so, or pass the question on). With it, the agent finds `get_store_hours` and answers. **The agent's code did not change.** In a real company, a different team can ship a new tool to your agent by updating their server.

## 4. Your server in other apps
Because the server follows the standard, apps you did not write can use it too. For example, Claude Desktop and many code editors are configured with a small JSON file that says how to **start** the server. This is an example (not run here):

```json
{
  "mcpServers": {
    "shop": {
      "command": "python",
      "args": ["MCP Tool Servers/shop_server.py"]
    }
  }
}
```

That is the whole integration: one entry that says how to start the server.

## What you learned
- The same server worked with **our own loop** and with the **Agents SDK**, with no change.
- The agent app has **no tool code**: it discovers tools at run time, so a server can grow without the app changing.
- Rules in the server (the $100 limit) hold for **every** client.

**Next:** the part that people forget. An MCP server is code from someone else, and its descriptions are text your model trusts.